# Duplication Questions

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

## Que1: Call Center Performance Metrics

**Difficulty:** Easy

### Problem

Build a daily call-volume summary for the support floor.

You are a data analyst at a telecommunications company. The call-center operations lead wants a per-day summary of how many distinct customers called in and how much total talk time was logged, so staffing can be matched to demand. Note that this raw feed lands with every column stored as text, so numeric values must be cast before aggregating.

Write a query to summarize calls per day. Join `cc_calls` to `cc_customer` on `cust_id` so that only calls from known customers are counted. Group by date and compute: the number of distinct customers who called that day (`num_customers`), and the total call duration that day (`total_duration`), casting duration from text to integer before summing. Return both aggregates as integers and sort the results by date in ascending order.

**Schema columns:** `cc_calls.call_id`, `cc_calls.cust_id`, `cc_calls.date`, `cc_calls.duration`, `cc_customer.cust_id`, `cc_customer.name`, `cc_customer.state`, `cc_customer.tenure`, `cc_customer.occupation`

**Output columns:** `date`, `num_customers`, `total_duration`

### Examples

#### Example 1

**Input:**

**cc_calls:**

| call_id | cust_id | date | duration |
|--------:|--------:|------|----------|
| 1 | 1 | 2022-01-01 | 100 |
| 2 | 2 | 2022-01-01 | 200 |
| 3 | 1 | 2022-01-02 | 150 |
| 4 | 3 | 2022-01-02 | 300 |
| 5 | 2 | 2022-01-03 | 50 |

**cc_customer:**

| cust_id | name | state | tenure | occupation |
|--------:|------|-------|-------:|------------|
| 1 | Alice | NY | 10 | doctor |
| 2 | Bob | CA | 12 | lawyer |
| 3 | Charlie | TX | 6 | engineer |

**Output:**

| date | num_customers | total_duration |
|------------|-------------:|---------------:|
| 2022-01-01 | 2 | 300 |
| 2022-01-02 | 2 | 450 |
| 2022-01-03 | 1 | 50 |

**Explanation:** On 2022-01-01, customers 1 and 2 called (2 distinct customers) for 100 + 200 = 300 total duration. On 2022-01-03 only customer 2 called, giving 1 distinct customer and 50 total duration. Days are listed in ascending date order.

### Constraints

- All columns are stored as text; cast `duration` to integer before summing.
- Inner join `cc_calls` to `cc_customer` on `cust_id` so only calls from known customers count.
- `num_customers` counts distinct customers per day, not total calls.
- Return `num_customers` and `total_duration` as integers.
- Output columns must be exactly `date`, `num_customers`, `total_duration`.
- Sort results by date in ascending order.

In [0]:
cc_calls_data = [("1","1","2022-01-01","100"),("2","2","2022-01-01","200"),("3","1","2022-01-02","150"),("4","3","2022-01-02","300"),("5","2","2022-01-03","50")]
cc_calls_df = spark.createDataFrame(cc_calls_data, ["call_id","cust_id","date","duration"])
cc_calls_df = cc_calls_df.withColumn("duration", col("duration").cast("int"))

cc_customer_data = [("1","Alice","NY","10","doctor"),("2","Bob","CA","12","lawyer"),("3","Charlie","TX","6","engineer")]
cc_customer_df = spark.createDataFrame(cc_customer_data, ["cust_id","name","state","tenure","occupation"])

display(cc_calls_df)
display(cc_customer_df)


joined_df = cc_calls_df.alias("cl").join(cc_customer_df.alias("cu"), on=col("cl.cust_id") == col("cu.cust_id"))
display(joined_df)

grouped_df = (
joined_df.groupBy("cl.date").agg(
    countDistinct("cu.cust_id").alias("num_customers"),
    sum("duration").alias("total_duration")
)
.orderBy("date")
)

display(grouped_df)



call_id,cust_id,date,duration
1,1,2022-01-01,100
2,2,2022-01-01,200
3,1,2022-01-02,150
4,3,2022-01-02,300
5,2,2022-01-03,50


cust_id,name,state,tenure,occupation
1,Alice,NY,10,doctor
2,Bob,CA,12,lawyer
3,Charlie,TX,6,engineer


call_id,cust_id,date,duration,cust_id,name,state,tenure,occupation
1,1,2022-01-01,100,1,Alice,NY,10,doctor
2,2,2022-01-01,200,2,Bob,CA,12,lawyer
3,1,2022-01-02,150,1,Alice,NY,10,doctor
4,3,2022-01-02,300,3,Charlie,TX,6,engineer
5,2,2022-01-03,50,2,Bob,CA,12,lawyer


date,num_customers,total_duration
2022-01-01,2,300
2022-01-02,2,450
2022-01-03,1,50


## Que2: Distinct Subject Count per Teacher

**Difficulty:** Easy

### Problem

A university tracks which subjects each teacher teaches across its departments. A teacher can teach the same subject in more than one department, but that subject should count only once for that teacher.

For every teacher, report how many distinct subjects they teach. Return one row per teacher with `teacher_id` and `unique_subject_count`, the number of different subjects that teacher teaches. Return the result table in any order.

**Schema columns:** `us_teacher_table.teacher_id`, `us_teacher_table.subject_id`, `us_teacher_table.dept_id`

**Output columns:** `teacher_id`, `unique_subject_count`

### Examples

#### Example 1

**Input:**

**us_teacher_table:**

| teacher_id | subject_id | dept_id |
|-----------:|-----------:|--------:|
| 1 | 2 | 3 |
| 1 | 2 | 4 |
| 1 | 3 | 3 |
| 2 | 1 | 1 |
| 2 | 4 | 2 |
| 3 | 2 | 5 |

**Output:**

| teacher_id | unique_subject_count |
|-----------:|---------------------:|
| 1 | 2 |
| 2 | 2 |
| 3 | 1 |

**Explanation:** Teacher 1 has three rows but only subjects 2 and 3; subject 2 appears twice (departments 3 and 4) so it counts once, giving 2 distinct subjects. Teacher 2 teaches subjects 1 and 4, giving 2, while teacher 3 teaches only subject 2, giving 1.

### Constraints

- A subject taught by the same teacher in multiple departments counts once for that teacher.
- Every teacher that appears in the table has at least one row.
- Return results matching the expected output schema and order.

In [0]:
us_teacher_table_data = [(1,2,3),(1,2,4),(1,3,3),(2,1,1),(2,4,2),(3,2,5)]
us_teacher_table_df = spark.createDataFrame(us_teacher_table_data, ["teacher_id","subject_id","dept_id"])
display(us_teacher_table_df)

output_df = us_teacher_table_df.groupBy("teacher_id").agg(countDistinct("subject_id").alias("unique_subject_count"))

display(output_df)


teacher_id,subject_id,dept_id
1,2,3
1,2,4
1,3,3
2,1,1
2,4,2
3,2,5


teacher_id,unique_subject_count
1,2
2,2
3,1


## Que3: Historical Sales Record Analysis

**Difficulty:** Easy

### Problem

A store logs every product it sells on each day. For each sell date, report how many distinct products were sold that day and the list of those product names.

The product list should contain each product once, sorted alphabetically and joined by a comma. Order the result by sell date, earliest first.

**Schema columns:** `sra_activities.sell_date`, `sra_activities.product`

**Output columns:** `sell_date`, `num_sold`, `products`

Sort the results by `sell_date`.

### Examples

#### Example 1

**Input:**

**sra_activities:**

| sell_date | product |
|-----------|----------|
| 2020-05-30 | Headphone |
| 2020-06-01 | Pencil |
| 2020-06-02 | Mask |
| 2020-05-30 | Basketball |
| 2020-06-01 | Bible |
| 2020-06-02 | Mask |
| 2020-05-30 | T-Shirt |

**Output:**

| sell_date | num_sold | products |
|-----------|--------:|----------|
| 2020-05-30 | 3 | Basketball,Headphone,T-Shirt |
| 2020-06-01 | 2 | Bible,Pencil |
| 2020-06-02 | 1 | Mask |

**Explanation:** On 2020-05-30 three distinct products were sold (Basketball, Headphone, T-Shirt), so `num_sold` is 3 and the names are listed alphabetically. On 2020-06-02 Mask appears twice but counts once, giving `num_sold` 1 and a single product name.

### Constraints

- Product names in `products` are distinct and sorted alphabetically, joined by a comma with no spaces.
- `num_sold` counts distinct products for that date.
- Return results matching the expected output schema and order.

In [0]:
sra_activities_data = [("2020-05-30", "Headphone"),("2020-06-01", "Pencil"),("2020-06-02", "Mask"),("2020-05-30", "Basketball"),("2020-06-01", "Bible"),("2020-06-02", "Mask"),("2020-05-30", "T-Shirt")]
sra_activities_df = spark.createDataFrame(sra_activities_data, ["sell_date", "product"])
display(sra_activities_df)

output_df = (
sra_activities_df.groupBy("sell_date").agg(
    countDistinct("product").alias("num_solds"),
    # string_agg_distinct(col("product"), ",").alias("products")
    concat_ws(",", sort_array(collect_list(col("product"))))
))

display(output_df)


sell_date,product
2020-05-30,Headphone
2020-06-01,Pencil
2020-06-02,Mask
2020-05-30,Basketball
2020-06-01,Bible
2020-06-02,Mask
2020-05-30,T-Shirt


sell_date,num_solds,"concat_ws(,, sort_array(collect_list(product), true))"
2020-05-30,3,"Basketball,Headphone,T-Shirt"
2020-06-01,2,"Bible,Pencil"
2020-06-02,1,"Mask,Mask"


## Que4: Delete Duplicate Emails (Keep Min ID)

**Difficulty:** Medium

### Problem

You are given a `person` table where the same email address can appear on more than one row. Return one row for each unique email address, keeping the row that has the smallest `id` and discarding the other rows that share that email.

**Schema columns:** `person.id`, `person.email`

**Output columns:**
- `id`: the smallest id among all rows sharing this email
- `email`: the unique email address

Output sorted by `id` ascending.

### Examples

#### Example 1

**Input:**

**person:**

| id | email |
|---:|-------|
| 1 | john@example.com |
| 2 | bob@example.com |
| 3 | john@example.com |
| 4 | john@another.com |
| 5 | bob@example.com |
| 6 | alice@example.com |

**Output:**

| id | email |
|---:|-------|
| 1 | john@example.com |
| 2 | bob@example.com |
| 4 | john@another.com |
| 6 | alice@example.com |

**Explanation:** `john@example.com` appears on ids 1 and 3, so only id 1 is kept. `bob@example.com` appears on ids 2 and 5, so only id 2 is kept. `john@another.com` (id 4) and `alice@example.com` (id 6) each occur once, so both are kept unchanged.

### Constraints

- For each email, keep only the row with the minimum `id`.
- Email matching is case-sensitive.
- Output sorted by `id` ascending.
- Return results matching the expected output schema and order.

In [0]:
person_data = [(1, "john@example.com"),(2, "bob@example.com"),(3, "john@example.com"),(4, "john@another.com"),(5, "bob@example.com"),(6, "alice@example.com")]

person_df = spark.createDataFrame(person_data, ["id", "email"])
display(person_df)

window_spec = Window.partitionBy("email").orderBy("id")

ranked_df = (
person_df.withColumn("rank", row_number().over(window_spec))
)

output_df = ranked_df.filter("rank = 1").drop("rank").orderBy("id")

display(output_df)


id,email
1,john@example.com
2,bob@example.com
3,john@example.com
4,john@another.com
5,bob@example.com
6,alice@example.com


id,email
1,john@example.com
2,bob@example.com
4,john@another.com
6,alice@example.com


## Que5: Data Pipeline Design Pattern (Merge/Upsert)

**Difficulty:** Hard

### Problem

A data pipeline merges an incremental `source_data` feed into an existing `target_data` table, keyed by `id`.

For every `id` that appears in either table, produce one merged record. When an `id` exists in both tables, keep the record with the later `updated_at`; ties and target-newer cases keep the target record. Return one row per `id` with columns `id`, `name`, `value`, `updated_at`, and `record_source`, where:
- `record_source` is `new` when the id appears only in the source
- `record_source` is `updated` when it appears in both and the source is newer
- `record_source` is `unchanged` when it appears only in the target or the target's `updated_at` is greater than or equal to the source's

The `name`, `value`, and `updated_at` come from whichever record was kept.

**Schema columns:** `source_data.id`, `source_data.name`, `source_data.value`, `source_data.updated_at`, `target_data.id`, `target_data.name`, `target_data.value`, `target_data.updated_at`

**Output columns:** `id`, `name`, `value`, `updated_at`, `record_source`

Sort the results by `id`.

### Examples

#### Example 1

**Input:**

**source_data:**

| id | name | value | updated_at |
|---:|------|------:|------------|
| 1 | Alice | 100 | 2024-01-10 |
| 2 | Bob | 200 | 2024-01-12 |
| 4 | David | 400 | 2024-01-14 |
| 5 | Eve | 500 | 2024-01-15 |

**target_data:**

| id | name | value | updated_at |
|---:|------|------:|------------|
| 1 | Alice | 95 | 2024-01-08 |
| 2 | Bob | 210 | 2024-01-15 |
| 3 | Charlie | 300 | 2024-01-13 |

**Output:**

| id | name | value | updated_at | record_source |
|---:|------|------:|------------|---------------|
| 1 | Alice | 100 | 2024-01-10 | updated |
| 2 | Bob | 210 | 2024-01-15 | unchanged |
| 3 | Charlie | 300 | 2024-01-13 | unchanged |
| 4 | David | 400 | 2024-01-14 | new |
| 5 | Eve | 500 | 2024-01-15 | new |

**Explanation:** id 1 exists in both tables and the source `updated_at` 2024-01-10 is later than the target's 2024-01-08, so the source record wins and `record_source` is `updated`. id 2 also exists in both, but the target's 2024-01-15 is later than the source's 2024-01-12, so the target keeps value 210 and stays `unchanged`. ids 4 and 5 appear only in the source, so they are carried over as `new`.

### Constraints

- When an id appears in both tables, the record with the later `updated_at` is kept; on a tie or when the target is newer, the target record is kept.
- `record_source` is exactly one of `new`, `updated`, or `unchanged`.
- Running the merge again on its own output produces the same result.
- Return results matching the expected output schema and order.

In [0]:
source_data_data = [(1, "Alice", 100, "2024-01-10"),(2, "Bob", 200, "2024-01-12"),(4, "David", 400, "2024-01-14"),(5, "Eve", 500, "2024-01-15")]
source_data_df = spark.createDataFrame(source_data_data, ["id", "name", "value", "updated_at"])
source_data_df = source_data_df.withColumn("updated_at", to_date(col("updated_at")))

target_data_data = [(1, "Alice", 95, "2024-01-08"),(2, "Bob", 210, "2024-01-15"),(3, "Charlie", 300, "2024-01-13")]
target_data_df = spark.createDataFrame(target_data_data, ["id", "name", "value", "updated_at"])
target_data_df = target_data_df.withColumn("updated_at", to_date(col("updated_at")))

display(source_data_df)
display(target_data_df)

joined_df = source_data_df.alias("s").join(target_data_df.alias("t"), on=col("s.id") == col("t.id"), how="full")

flaged_df = (
joined_df.
    withColumn("record_source", 
         when(col("t.id").isNull(), "new")
        .when(col("s.updated_at") > col("t.updated_at"), "updated")
        .otherwise("unchanged")
    )
)

new_updated_records = flaged_df.filter((col("record_source") == "new") | (col("record_source") == "updated")).select("s.*", "record_source")
unchanged_records = flaged_df.filter(col("record_source") == "unchanged").select("t.*", "record_source")

output_df = new_updated_records.union(unchanged_records).orderBy("id")

display(output_df)

id,name,value,updated_at
1,Alice,100,2024-01-10
2,Bob,200,2024-01-12
4,David,400,2024-01-14
5,Eve,500,2024-01-15


id,name,value,updated_at
1,Alice,95,2024-01-08
2,Bob,210,2024-01-15
3,Charlie,300,2024-01-13


id,name,value,updated_at,record_source
1,Alice,100,2024-01-10,updated
2,Bob,210,2024-01-15,unchanged
3,Charlie,300,2024-01-13,unchanged
4,David,400,2024-01-14,new
5,Eve,500,2024-01-15,new


## Que6: Data Consistency and Idempotency

**Difficulty:** Hard

### Problem

A data pipeline merges a batch of incoming records into a table of existing records. For each entity, compare the most recent incoming record against the most recent existing record and return one row per incoming entity:

- `entity_id` — the entity identifier
- `final_value` — the resolved value
- `source_system` — source system of the record that provides the final value
- `last_updated` — timestamp of the record that provides the final value, returned as a string formatted `YYYY-MM-DD HH24:MI:SS`
- `status`:
  - `new` if the entity appears only in incoming
  - `duplicate` if the entity's latest incoming value matches its latest existing value
  - `updated` if the entity exists in both but the latest incoming value differs

When a value is **updated**, the incoming record supplies the final value, source system, and timestamp. When it is a **duplicate**, keep whichever of the two records is later by timestamp (the incoming record wins on a tie). Entities that appear only in existing records are **not** returned.

**Schema columns:** `existing_records.record_id`, `existing_records.source_system`, `existing_records.entity_id`, `existing_records.value`, `existing_records.loaded_at`, `incoming_records.record_id`, `incoming_records.source_system`, `incoming_records.entity_id`, `incoming_records.value`, `incoming_records.received_at`

**Output columns:** `entity_id`, `final_value`, `source_system`, `last_updated`, `status`

Order the result by `entity_id` ascending.

### Examples

#### Example 1

**Input:**

**existing_records:**

| record_id | source_system | entity_id | value | loaded_at |
|-----------|--------------|----------:|-------|--------------------|
| REC_0001 | system_a | 6002 | value_273 | 2024-02-10 00:00:00 |
| REC_0003 | system_a | 6004 | value_496 | 2024-01-06 00:00:00 |
| REC_0004 | system_b | 6005 | value_226 | 2024-02-19 00:00:00 |

**incoming_records:**

| record_id | source_system | entity_id | value | received_at |
|-----------|--------------|----------:|-------|--------------------|
| NEW_0001 | system_c | 7002 | new_value_287 | 2024-02-28 00:00:00 |
| REC_0003 | system_a | 6004 | value_496 | 2024-01-06 00:00:00 |
| REC_0003 | system_a | 6004 | updated_value_395 | 2024-02-27 00:00:00 |
| REC_0004 | system_b | 6005 | value_226 | 2024-02-19 00:00:00 |

**Output:**

| entity_id | final_value | source_system | last_updated | status |
|----------:|-------------|--------------|---------------------|----------|
| 6004 | updated_value_395 | system_a | 2024-02-27 00:00:00 | updated |
| 6005 | value_226 | system_b | 2024-02-19 00:00:00 | duplicate |
| 7002 | new_value_287 | system_c | 2024-02-28 00:00:00 | new |

**Explanation:** Entity 6004's latest incoming record is `updated_value_395` at 2024-02-27, which differs from its existing value `value_496`, so it is `updated` and takes the incoming value, `system_a`, and timestamp 2024-02-27. Entity 6005's latest incoming value `value_226` matches its existing value, so it is a `duplicate`. Entity 7002 has no existing record, so it is `new`. Entity 6002 exists only in existing records and is dropped.

### Constraints

- Compare the most recent incoming record with the most recent existing record for each entity.
- Same entity, same value = `duplicate`; same entity, different value = `updated`; incoming-only entity = `new`.
- On an update, the incoming record supplies the final value, source system, and timestamp.
- On a duplicate, keep the later record by timestamp; the incoming record wins on a tie.
- Entities present only in existing records are not returned.
- `last_updated` is returned as a string formatted `YYYY-MM-DD HH24:MI:SS`.
- Return results matching the expected output schema and order.

In [0]:
existing_records_df_data = [("REC_0001", "system_a", 6002, "value_273", "2024-02-10 00:00:00"),("REC_0003", "system_a", 6004, "value_496", "2024-01-06 00:00:00"),("REC_0004", "system_b", 6005, "value_226", "2024-02-19 00:00:00")]
existing_records_df_df = spark.createDataFrame(existing_records_df_data, ["record_id", "source_system", "entity_id", "value", "loaded_at"])
existing_records_df_df = existing_records_df_df.withColumn("loaded_at", to_date(col("loaded_at")))

incoming_records_df_data = [("NEW_0001", "system_c", 7002, "new_value_287", "2024-02-28 00:00:00"),("REC_0003", "system_a", 6004, "value_496", "2024-01-06 00:00:00"),("REC_0003", "system_a", 6004, "updated_value_395", "2024-02-27 00:00:00"),("REC_0004", "system_b", 6005, "value_226", "2024-02-19 00:00:00")]
incoming_records_df_df = spark.createDataFrame(incoming_records_df_data, ["record_id", "source_system", "entity_id", "value", "received_at"])
incoming_records_df_df = incoming_records_df_df.withColumn("received_at", to_date(col("received_at")))

display(existing_records_df_df)
display(incoming_records_df_df)

incoming_window = (
    Window
    .partitionBy("entity_id")
    .orderBy(col("received_at").desc())
)

latest_incoming_df = (
    incoming_records_df
    .withColumn("rn", row_number().over(incoming_window))
    .filter(col("rn") == 1)
    .drop("rn")
)

existing_window = (
    Window
    .partitionBy("entity_id")
    .orderBy(col("loaded_at").desc())
)

latest_existing_df = (
    existing_records_df
    .withColumn("rn", row_number().over(existing_window))
    .filter(col("rn") == 1)
    .drop("rn")
)

joined_df = (
    latest_incoming_df.alias("i")
    .join(
        latest_existing_df.alias("e"),
        col("i.entity_id") == col("e.entity_id"),
        "left"
    )
)

result_df = (
    joined_df
    .withColumn(
        "status",
        when(col("e.entity_id").isNull(), "new")
        .when(col("i.value") == col("e.value"), "duplicate")
        .otherwise("updated")
    )
)

result_df = (
    result_df
    .withColumn(
        "final_value",
        when(
            col("status").isin("new", "updated"),
            col("i.value")
        )
        .otherwise(
            when(
                col("i.received_at") >= col("e.loaded_at"),
                col("i.value")
            ).otherwise(col("e.value"))
        )
    )
    .withColumn(
        "final_source_system",
        when(
            col("status").isin("new", "updated"),
            col("i.source_system")
        )
        .otherwise(
            when(
                col("i.received_at") >= col("e.loaded_at"),
                col("i.source_system")
            ).otherwise(col("e.source_system"))
        )
    )
    .withColumn(
        "last_updated",
        when(
            col("status").isin("new", "updated"),
            date_format(col("i.received_at"), "yyyy-MM-dd HH:mm:ss")
        )
        .otherwise(
            when(
                col("i.received_at") >= col("e.loaded_at"),
                date_format(col("i.received_at"), "yyyy-MM-dd HH:mm:ss")
            ).otherwise(
                date_format(col("e.loaded_at"), "yyyy-MM-dd HH:mm:ss")
            )
        )
    )
)

output_df = (
    result_df
    .select(
        col("i.entity_id").alias("entity_id"),
        col("final_value"),
        col("final_source_system").alias("source_system"),
        col("last_updated"),
        col("status")
    )
    .orderBy("entity_id")
)

display(output_df)






record_id,source_system,entity_id,value,loaded_at
REC_0001,system_a,6002,value_273,2024-02-10
REC_0003,system_a,6004,value_496,2024-01-06
REC_0004,system_b,6005,value_226,2024-02-19


record_id,source_system,entity_id,value,received_at
NEW_0001,system_c,7002,new_value_287,2024-02-28
REC_0003,system_a,6004,value_496,2024-01-06
REC_0003,system_a,6004,updated_value_395,2024-02-27
REC_0004,system_b,6005,value_226,2024-02-19


record_id,source_system,entity_id,value,received_at,record_id,source_system,entity_id,value,loaded_at,status
null,null,null,null,null,REC_0001,system_a,6002,value_273,2024-02-10,updated
REC_0003,system_a,6004,updated_value_395,2024-02-27,REC_0003,system_a,6004,value_496,2024-01-06,updated
REC_0004,system_b,6005,value_226,2024-02-19,REC_0004,system_b,6005,value_226,2024-02-19,duplicate
NEW_0001,system_c,7002,new_value_287,2024-02-28,null,null,null,null,null,new
